# Distribuições do alvo (Y), do atributo sensível (S) e conjunta P(S, Y)

Resposta à solicitação do revisor: para cada **base**, **gerador** e **fold**, calculamos

- proporção de cada classe de $Y$;
- proporção dos grupos protegido e privilegiado de $S$;
- contagem e proporção das quatro combinações $(S, Y)$;
- os mesmos valores para a **partição real de treinamento** de cada fold (referência);
- média e desvio-padrão entre os 5 folds.

**Convenções** (idênticas a `src/utils.run_clf` e `configs/datasets_config.json`):
$Y = 1$ sse `target_col == target_value`; $S = 1$ marca o grupo **protegido**
(`col == sensitive_value` quando `sensitive_value_type == 'Protected'`;
`col != sensitive_value` quando `'Privileged'`, caso do COMPAS, em que o grupo
protegido é não-caucasiano). Na base `bank_marketing` a coluna `age` já está
binarizada nos dados originais (grupo protegido = `age == 0`).

Os CSVs agregados são produzidos por `compute_sy_distributions.py` (nesta pasta)
e salvos em `outputs/`. Este notebook apenas recomputa/carrega e apresenta.

In [1]:
from pathlib import Path

import pandas as pd

import compute_sy_distributions as csd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

OUT_DIR = csd.OUT_DIR
per_fold_path = OUT_DIR / "sy_distributions_per_fold.csv"

if per_fold_path.exists():
    per_fold = pd.read_csv(per_fold_path)
else:
    per_fold = csd.compute_distributions()
    per_fold.to_csv(per_fold_path, index=False)

summary = csd.summarize(per_fold)
summary.to_csv(OUT_DIR / "sy_distributions_summary.csv", index=False)
print(f"{len(per_fold)} linhas (base x gerador x fold); CSVs em {OUT_DIR}")

Matplotlib is building the font cache; this may take a moment.


140 linhas (base x gerador x fold); CSVs em /Users/felipebfg/Documents/Msc_Final_copy/review_analysis/outputs


## Tabela por base × gerador (média ± desvio-padrão entre folds)

Colunas: $P(Y{=}1)$, $P(S{=}\text{prot})$ e as quatro células da conjunta
$P(S, Y)$ — `s1` = grupo protegido, `s0` = privilegiado; `y1` = classe positiva.
A linha `real_train` é a partição real de treinamento usada como referência.

In [3]:
def fmt_mean_std(df, col):
    return (
        df[f"{col}_mean"].map("{:.4f}".format)
        + " ± "
        + df[f"{col}_std"].map("{:.4f}".format)
    )

cols = ["p_y1", "p_s_protected", "p_s0_y0", "p_s0_y1", "p_s1_y0", "p_s1_y1"]
table = summary[["dataset", "generator"]].copy()
for c in cols:
    table[c] = fmt_mean_std(summary, c)
table["n"] = summary["n_mean"].round(0).astype(int)

order = ["real_train"] + csd.GENERATORS
for dataset in csd.DATASETS:
    sub = (
        table[table["dataset"] == dataset]
        .set_index("generator")
        .reindex(order)
        .drop(columns="dataset")
    )
    print(f"==== {dataset} ====")
    display(sub)

==== adult ====


,p_y1,p_s_protected,p_s0_y0,p_s0_y1,p_s1_y0,p_s1_y1,n
generator,,,,,,,
real_train,0.2393 ± 0.0000,0.3309 ± 0.0025,0.4664 ± 0.0023,0.2027 ± 0.0005,0.2943 ± 0.0023,0.0366 ± 0.0005,29305
arf,0.2354 ± 0.0035,0.3351 ± 0.0060,0.4675 ± 0.0030,0.1974 ± 0.0038,0.2971 ± 0.0051,0.0380 ± 0.0024,29305
ctabgan,0.2524 ± 0.0399,0.3357 ± 0.0168,0.4548 ± 0.0312,0.2095 ± 0.0334,0.2928 ± 0.0203,0.0428 ± 0.0092,29305
ctgan,0.2502 ± 0.0100,0.3273 ± 0.0058,0.4629 ± 0.0119,0.2098 ± 0.0099,0.2869 ± 0.0069,0.0404 ± 0.0015,29305
ddpm,0.2309 ± 0.0067,0.3267 ± 0.0064,0.4748 ± 0.0088,0.1984 ± 0.0052,0.2943 ± 0.0044,0.0324 ± 0.0024,29305
realtabformer,0.2599 ± 0.0393,0.3142 ± 0.0285,0.4654 ± 0.0349,0.2204 ± 0.0336,0.2747 ± 0.0340,0.0395 ± 0.0079,29305
tvae,0.2452 ± 0.0095,0.3052 ± 0.0130,0.4891 ± 0.0047,0.2057 ± 0.0097,0.2658 ± 0.0132,0.0394 ± 0.0049,29305


==== bank_marketing ====


,p_y1,p_s_protected,p_s0_y0,p_s0_y1,p_s1_y0,p_s1_y1,n
generator,,,,,,,
real_train,0.1170 ± 0.0000,0.0571 ± 0.0009,0.8436 ± 0.0009,0.0994 ± 0.0004,0.0394 ± 0.0009,0.0176 ± 0.0004,27126
arf,0.1071 ± 0.0008,0.0512 ± 0.0020,0.8547 ± 0.0014,0.0941 ± 0.0020,0.0382 ± 0.0007,0.0130 ± 0.0016,27126
ctabgan,0.3516 ± 0.0969,0.0934 ± 0.0250,0.6086 ± 0.1009,0.2980 ± 0.0874,0.0398 ± 0.0129,0.0537 ± 0.0136,27126
ctgan,0.1226 ± 0.0167,0.0596 ± 0.0122,0.8371 ± 0.0230,0.1033 ± 0.0112,0.0403 ± 0.0083,0.0193 ± 0.0058,27126
ddpm,0.1040 ± 0.0033,0.0428 ± 0.0047,0.8661 ± 0.0076,0.0911 ± 0.0032,0.0299 ± 0.0046,0.0129 ± 0.0004,27126
realtabformer,0.1341 ± 0.0236,0.0464 ± 0.0055,0.8370 ± 0.0245,0.1166 ± 0.0201,0.0289 ± 0.0027,0.0175 ± 0.0036,27126
tvae,0.1253 ± 0.0242,0.0419 ± 0.0027,0.8497 ± 0.0219,0.1084 ± 0.0230,0.0251 ± 0.0026,0.0169 ± 0.0022,27126


==== compas ====


,p_y1,p_s_protected,p_s0_y0,p_s0_y1,p_s1_y0,p_s1_y1,n
generator,,,,,,,
real_train,0.4507 ± 0.0001,0.6589 ± 0.0071,0.2086 ± 0.0047,0.1325 ± 0.0036,0.3406 ± 0.0048,0.3182 ± 0.0036,4328
arf,0.4778 ± 0.0058,0.6595 ± 0.0054,0.1950 ± 0.0062,0.1455 ± 0.0036,0.3272 ± 0.0018,0.3323 ± 0.0060,4328
ctabgan,0.5497 ± 0.1124,0.7415 ± 0.0484,0.1395 ± 0.0544,0.1190 ± 0.0256,0.3108 ± 0.0603,0.4307 ± 0.1048,4328
ctgan,0.4851 ± 0.0172,0.6690 ± 0.0224,0.1851 ± 0.0155,0.1460 ± 0.0082,0.3299 ± 0.0091,0.3391 ± 0.0218,4328
ddpm,0.4822 ± 0.0036,0.6592 ± 0.0077,0.1909 ± 0.0082,0.1499 ± 0.0057,0.3268 ± 0.0066,0.3324 ± 0.0057,4328
realtabformer,0.4185 ± 0.0451,0.6734 ± 0.0400,0.2133 ± 0.0329,0.1133 ± 0.0222,0.3683 ± 0.0294,0.3051 ± 0.0385,4328
tvae,0.4788 ± 0.0154,0.6634 ± 0.0307,0.1902 ± 0.0270,0.1464 ± 0.0182,0.3310 ± 0.0159,0.3324 ± 0.0247,4328


==== german ====


,p_y1,p_s_protected,p_s0_y0,p_s0_y1,p_s1_y0,p_s1_y1,n
generator,,,,,,,
real_train,0.3000 ± 0.0000,0.3040 ± 0.0164,0.5003 ± 0.0111,0.1957 ± 0.0074,0.1997 ± 0.0111,0.1043 ± 0.0074,600
arf,0.2937 ± 0.0114,0.2963 ± 0.0254,0.5047 ± 0.0149,0.1990 ± 0.0182,0.2017 ± 0.0195,0.0947 ± 0.0092,600
ctabgan,0.2337 ± 0.0751,0.2530 ± 0.0963,0.5923 ± 0.1230,0.1547 ± 0.0649,0.1740 ± 0.0632,0.0790 ± 0.0400,600
ctgan,0.3450 ± 0.0358,0.3290 ± 0.0442,0.4513 ± 0.0280,0.2197 ± 0.0331,0.2037 ± 0.0250,0.1253 ± 0.0327,600
ddpm,0.2707 ± 0.0315,0.3650 ± 0.0407,0.4757 ± 0.0475,0.1593 ± 0.0174,0.2537 ± 0.0555,0.1113 ± 0.0454,600
realtabformer,0.3233 ± 0.0414,0.2853 ± 0.0508,0.4940 ± 0.0300,0.2207 ± 0.0392,0.1827 ± 0.0243,0.1027 ± 0.0419,600
tvae,0.3320 ± 0.0140,0.2877 ± 0.0627,0.4907 ± 0.0307,0.2217 ± 0.0380,0.1773 ± 0.0344,0.1103 ± 0.0303,600


## Contagens das quatro combinações (S, Y) por fold

Tabela completa por base × gerador × fold, com contagens e proporções
(disponível também em `outputs/sy_distributions_per_fold.csv`).

In [4]:
count_cols = [
    "dataset", "generator", "fold", "n",
    "count_s0_y0", "count_s0_y1", "count_s1_y0", "count_s1_y1",
    "p_s0_y0", "p_s0_y1", "p_s1_y0", "p_s1_y1",
]
per_fold[count_cols].round(4)

,dataset,generator,fold,n,count_s0_y0,count_s0_y1,count_s1_y0,count_s1_y1,p_s0_y0,p_s0_y1,p_s1_y0,p_s1_y1
0,adult,real_train,0,29304,13616,5954,8676,1058,0.4646,0.2032,0.2961,0.0361
1,adult,arf,0,29304,13737,5720,8804,1043,0.4688,0.1952,0.3004,0.0356
2,adult,ctabgan,0,29304,14749,4850,8726,979,0.5033,0.1655,0.2978,0.0334
3,adult,ctgan,0,29304,13430,6090,8674,1110,0.4583,0.2078,0.2960,0.0379
4,adult,ddpm,0,29304,13618,6068,8602,1016,0.4647,0.2071,0.2935,0.0347
...,...,...,...,...,...,...,...,...,...,...,...,...
135,german,ctabgan,4,600,464,86,39,11,0.7733,0.1433,0.0650,0.0183
136,german,ctgan,4,600,268,152,121,59,0.4467,0.2533,0.2017,0.0983
137,german,ddpm,4,600,324,95,133,48,0.5400,0.1583,0.2217,0.0800
138,german,realtabformer,4,600,316,131,108,45,0.5267,0.2183,0.1800,0.0750


## Figuras: conjunta P(S, Y) — real vs. sintético por gerador

Barras = média entre os 5 folds; barras de erro = desvio-padrão.
As figuras também são salvas como `outputs/fig_joint_sy_<base>.png`.

In [5]:
import matplotlib.pyplot as plt
import numpy as np

cells_ = ["p_s0_y0", "p_s0_y1", "p_s1_y0", "p_s1_y1"]
labels = [
    "P(S=priv, Y=0)", "P(S=priv, Y=1)",
    "P(S=prot, Y=0)", "P(S=prot, Y=1)",
]
order = ["real_train"] + csd.GENERATORS

for dataset in csd.DATASETS:
    sub = per_fold[per_fold["dataset"] == dataset]
    mean = sub.groupby("generator")[cells_].mean().reindex(order)
    std = sub.groupby("generator")[cells_].std().reindex(order)
    fig, ax = plt.subplots(figsize=(10, 4.5))
    x = np.arange(len(order))
    width = 0.2
    for i, (cell, label) in enumerate(zip(cells_, labels)):
        ax.bar(x + (i - 1.5) * width, mean[cell], width,
               yerr=std[cell], capsize=3, label=label)
    ax.set_xticks(x)
    ax.set_xticklabels(order, rotation=20)
    ax.set_ylabel("proporção (média ± dp, 5 folds)")
    ax.set_title(f"Conjunta P(S, Y): real vs. sintético — {dataset}")
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(csd.OUT_DIR / f"fig_joint_sy_{dataset}.png", dpi=150)
    plt.show()

/var/folders/0b/0r61hs8n7zqbyzpd1yvr3dwr0000gn/T/ipykernel_95561/4068657164.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/0b/0r61hs8n7zqbyzpd1yvr3dwr0000gn/T/ipykernel_95561/4068657164.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/0b/0r61hs8n7zqbyzpd1yvr3dwr0000gn/T/ipykernel_95561/4068657164.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/0b/0r61hs8n7zqbyzpd1yvr3dwr0000gn/T/ipykernel_95561/4068657164.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
